In [63]:
import pandas as pd
from pymongo import MongoClient
import datetime
import os
from dotenv import load_dotenv
import random

# Load environment variables
load_dotenv()

# MongoDB connection setup
MONGO_CONNECTION_STRING = os.getenv("MONGO_CONNECTION_STRING")
MONGO_DATABASE_NAME = os.getenv("MONGO_DATABASE_NAME")

client = MongoClient(MONGO_CONNECTION_STRING)
db = client[MONGO_DATABASE_NAME]
sender_names_collection = db.sender_names

ConfigurationError: The resolution lifetime expired after 21.142 seconds: Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.; Server Do53:192.168.1.1@53 answered The DNS operation timed out.

In [ ]:
df1 = pd.read_excel("/Users/suphakorninsee/Desktop/CEDT/Intern/Second_Years/SMS_sender/Backend/mock_nbtc_responses/Dtac_ตารางรายงาน Call Center Voice + SMS 01072025).xlsx")
df2 = pd.read_excel("/Users/suphakorninsee/Desktop/CEDT/Intern/Second_Years/SMS_sender/Backend/mock_nbtc_responses/True_ตารางรายงาน Call Center Voice + SMS (29062025).xlsx")

sender_names = pd.concat([df1["หมายเลขที่แสดง/Sender Name"], df2["หมายเลขที่แสดง/Sender Name"]]).dropna().unique().tolist()

sender_names = [str(name).replace("_x000D_", "") for name in sender_names]

print(f"Found {len(sender_names)} unique sender names: {sender_names}")

Found 280 unique sender names: ['0956802491', 'MAWINLOGI', 'lewreath', 'dpss', '0933235210', '08942495612', '08907001634', '08965630626', '0619978659', '0805587631', '0991142573', '020919901', '0943352407', '0628825001', '0823693832', '0933235280', 'Connect', '0627608676', 'n2ntoner', '0933235245', '0634083893', '0804868255', '0JOYOFFICE', '0994139189', 'bonk', '0948394620', 'TrueMoney', '0967820142', 'ISPORTPR', 'NPEShop', 'CDVBAG', 'crazycase', '0946321814', '0994094106', '0944692306', '0994060329', '0659987012', '0659986992', '0659987035', '0659987029', '0659987043', '0659986994', '0659987008', '0659987046', '0659987018', '0659987049', '0659987021', '0659986983', '0659987050', '0659987013', '0659987027', '0943562920', '697106581274438', '0628275894', '0823901093', '0827125631', '697106584707730', '0612703390', '0827844957', '0612706820', '0627424552', '0949366379', '0619918629', '0620108914', '0827247674', '0619895210', '0990260843', '0619415927', '0975048674', '0985470051', '096625

In [ ]:
today_str = datetime.date.today().strftime("%Y-%m-%d")
updated_at = datetime.datetime.now()

# สร้าง dictionary เพื่อ map sender_name กับ mobile_provider จาก Excel
provider_map = {}
for df in [df1, df2]:
    for _, row in df.iterrows():
        sender_name = str(row["หมายเลขที่แสดง/Sender Name"]).replace("_x000D_", "")
        provider = str(row["โครงข่ายที่ใช้งาน(โครงข่ายต้นทาง)"]).lower()
        provider_map[sender_name] = provider

data = [
    {
        "sender_name": sender_names[i % len(sender_names)],
        "mobile_provider": provider_map.get(sender_names[i % len(sender_names)], "unknown"),
        "phone_number": f"08{random.randint(0, 9)}{i+1:07d}",
        "full_name": f"นายทดสอบ {i+1}",
        "date": today_str,
        "status": [],
        "created_at": updated_at,
        "updated_at": updated_at
    } for i in range(len(sender_names))
]

print(f"Generated {len(data)} mock data entries")

Generated 280 mock data entries


In [ ]:
if data:
    sender_names_collection.insert_many(data)
    print(f"Inserted {len(data)} mock senders into MongoDB")
    print("✅ เพิ่ม mock sender เข้า MongoDB แล้ว")
else:
    print("No data to insert")

Inserted 280 mock senders into MongoDB
✅ เพิ่ม mock sender เข้า MongoDB แล้ว


In [ ]:
dtac_count_df1 = df1[df1["โครงข่ายที่ใช้งาน(โครงข่ายต้นทาง)"].str.lower() == "dtac"].shape[0]

print(f"Number of rows with 'Dtac' in 'โครงข่ายที่ใช้งาน(โครงข่ายต้นทาง)' in Dtac file: {dtac_count_df1}")

Number of rows with 'Dtac' in 'โครงข่ายที่ใช้งาน(โครงข่ายต้นทาง)' in Dtac file: 61
